In [2]:
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import MACCSkeys
from rdkit.Chem import AllChem

from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error


In [3]:
DESCRIPTOR_LIST = Descriptors._descList  # list of (name, function)
DESC_NAMES = [name for name, _ in DESCRIPTOR_LIST]

def rdkit_descriptors_from_smiles(smiles: str) -> dict:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {name: np.nan for name in DESC_NAMES}
    return {name: func(mol) for name, func in DESCRIPTOR_LIST}


In [4]:
def morgan_bits(mol, radius=2, nBits=2048):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nBits)
    arr = np.zeros((nBits,), dtype=np.int8)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def maccs_bits(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)  # 167 bits (index 0..166)
    arr = np.zeros((len(fp),), dtype=np.int8)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def fingerprints_from_smiles(smiles: str, radius=2, nBits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return (np.full((nBits,), 0, dtype=np.int8),
                np.full((167,), 0, dtype=np.int8))
    return morgan_bits(mol, radius=radius, nBits=nBits), maccs_bits(mol)


In [5]:
def featurize_dataframe(df: pd.DataFrame, smiles_col="SMILES", radius=2, nBits=2048) -> pd.DataFrame:
    # 1) Descriptors
    desc_rows = [rdkit_descriptors_from_smiles(s) for s in df[smiles_col]]
    desc_df = pd.DataFrame(desc_rows)

    # 2) Fingerprints
    morgan_list = []
    maccs_list = []
    for s in df[smiles_col]:
        morgan_arr, maccs_arr = fingerprints_from_smiles(s, radius=radius, nBits=nBits)
        morgan_list.append(morgan_arr)
        maccs_list.append(maccs_arr)

    morgan_df = pd.DataFrame(np.vstack(morgan_list), columns=[f"ECFP_{i}" for i in range(nBits)])
    maccs_df  = pd.DataFrame(np.vstack(maccs_list),  columns=[f"MACCS_{i}" for i in range(167)])

    # zusammenführen (SMILES behalten, falls ihr später noch was wollt)
    out = pd.concat([df.reset_index(drop=True), desc_df, morgan_df, maccs_df], axis=1)
    return out



In [6]:
def make_ml_matrices(train_df, test_df, target_col="Tm"):
    # drop non-features
    X_train = train_df.drop(columns=["SMILES", target_col], errors="ignore")
    y_train = train_df[target_col].values

    X_test  = test_df.drop(columns=["SMILES", "id"], errors="ignore")

    # Imputer (wichtig, weil Descriptors manchmal NaNs liefern)
    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp  = imputer.transform(X_test)

    return X_train_imp, y_train, X_test_imp, imputer, X_train.columns


In [7]:
import xgboost as xgb
from sklearn.model_selection import KFold

def cv_xgb_mae(X, y, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    maes = []

    for tr, va in kf.split(X):
        Xtr, Xva = X[tr], X[va]
        ytr, yva = y[tr], y[va]

        model = xgb.XGBRegressor(
            n_estimators=5000,
            learning_rate=0.03,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="reg:absoluteerror",   # MAE direkt
            tree_method="hist",
            random_state=seed
        )

        model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False, early_stopping_rounds=200)
        pred = model.predict(Xva)
        maes.append(mean_absolute_error(yva, pred))

    return float(np.mean(maes)), float(np.std(maes))


In [ ]:
# 1) Load base data
train_base = pd.read_csv("melting-point-data/train.csv")[["SMILES", "Tm"]]
test_base  = pd.read_csv("melting-point-data/test.csv")[["id", "SMILES"]]

train_extended_df = pd.read_csv("trainExtended1812.csv")[["SMILES", "Tm"]]

# 2) Optional external merge:
# train_merged = load_and_merge_external(
#     "melting-point-data/train.csv",
#     bradley_xlsx="BradleyMeltingPointDataset.xlsx",
#     bradleyplus_xlsx="BradleyDoublePlusGoodMeltingPointDataset.xlsx"
# )
train_merged = train_base

# 3) Featurize
train_feat = featurize_dataframe(train_merged, radius=2, nBits=2048)
test_feat  = featurize_dataframe(test_base,  radius=2, nBits=2048)
train_extended = featurize_dataframe(train_extended_df)

# 4) Build matrices
X_train, y_train, X_test, imputer, feat_cols = make_ml_matrices(train_feat, test_feat)

# 5) CV check (optional, aber empfehlenswert)
# mean_mae, std_mae = cv_xgb_mae(X_train, y_train)
# print("CV MAE:", mean_mae, "+/-", std_mae)

#Train final + predict
model = xgb.XGBRegressor(
    n_estimators=8000,
    learning_rate=0.01,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:absoluteerror",
    tree_method="hist",
    random_state=42
)
model.fit(X_train, y_train, verbose=False)
pred_test = model.predict(X_test)

# 7) Submission
submission = pd.DataFrame({"id": test_base["id"], "Tm": pred_test})
# submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")


[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerator
[13:32:57] DEPRECATION WARNING: please use MorganGenerat

Saved submission.csv


In [9]:
df = train_feat.drop(["SMILES"], axis=1)

df.to_csv("melting-point-data/morganfeatures.csv")

In [14]:
test_feat.to_csv("melting-point-data/morganfeatures_test.csv")

In [11]:
# model = xgb.XGBRegressor(
#     n_estimators=5000,
#     learning_rate=0.03,
#     max_depth=6,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     reg_lambda=1.0,
#     objective="reg:absoluteerror",
#     tree_method="hist",
#     random_state=42
# )

In [7]:
train_extended_df = pd.read_csv("trainExtended1812(2).csv")[["SMILES", "Tm"]]

train_extended = featurize_dataframe(train_extended_df)


train_extended.to_csv("trainExtended_morgan1812(2).csv", index=False)

[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerator
[13:42:12] DEPRECATION WARNING: please use MorganGenerat